# 04 · Evaluate the text-span detector on the supplied merged dataset

This notebook consumes `data/merged_gliner/test_gliner.jsonl` and understands its schema: `tokenized_text` plus inclusive token-index spans in `ner`. It evaluates the configured ConsentShield detector (GLiNER when enabled and loaded, otherwise the deterministic validator fallback) without training or changing the detector.

The test file mixes `ai4privacy` and `consentshield_synthetic` records. To keep the optional model run practical, evaluation samples up to 500 records from each source by default. Increase `MAX_PER_SOURCE` or set it to `None` to evaluate all test records. Results are reported separately by source and canonical ConsentShield category; the report never stores raw text.

In [1]:
import json, random, time, platform, collections
from pathlib import Path
import numpy as np

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "data" / "merged_gliner" / "test_gliner.jsonl").exists()), HERE)
DATA = ROOT / "data" / "merged_gliner" / "test_gliner.jsonl"
ADV = ROOT / "data" / "adversarial" / "adversarial_cases.jsonl"
RESULTS = ROOT / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
import sys; sys.path.insert(0, str(ROOT))
try:
    from app.detectors.gliner_detector import GlinerConfig, build_text_detector, ValidatorTextDetector
except ImportError:
    from gliner_detector import GlinerConfig, build_text_detector, ValidatorTextDetector
if not DATA.exists(): raise FileNotFoundError(f"Dataset not found: {DATA}")
SEED = 20261017
MAX_PER_SOURCE = 500  # set None to evaluate the entire test split
print("Root:", ROOT, "| machine:", platform.machine())

Root: C:\Users\Chris\Downloads\ibmz2026-main\ibmz2026-main | machine: AMD64


## Load the current token-indexed test data

Gold spans use inclusive token indices. The helper below creates a joined text and converts token spans to character offsets for comparison with the detector API. This reconstructed text is used only for evaluation; it is not the original raw source string.

In [2]:
def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def reconstruct(row):
    tokens = [str(t) for t in (row.get("tokenized_text") or [])]
    starts, ends, pieces = [], [], []
    pos = 0
    for i, tok in enumerate(tokens):
        if i: pos += 1
        starts.append(pos); pieces.append(tok); pos += len(tok); ends.append(pos)
    text = " ".join(pieces)
    gold = []
    for span in row.get("ner") or []:
        if not isinstance(span, (list, tuple)) or len(span) != 3: continue
        a, b, label = int(span[0]), int(span[1]), str(span[2])
        if 0 <= a <= b < len(tokens):
            gold.append({"start": starts[a], "end": ends[b], "label": label})
    return text, gold

rows = read_jsonl(DATA)
by_source = collections.defaultdict(list)
for r in rows: by_source[r.get("source", "UNKNOWN")].append(r)
rng = random.Random(SEED)
selected = []
for source, items in sorted(by_source.items()):
    if MAX_PER_SOURCE is not None and len(items) > MAX_PER_SOURCE:
        items = rng.sample(items, MAX_PER_SOURCE)
    selected.extend(items)
print("Full test rows:", len(rows), "| selected:", len(selected), "| by source:", dict(collections.Counter(r.get("source", "UNKNOWN") for r in selected)))
print("Selected gold span counts:", dict(collections.Counter(s[2] for r in selected for s in (r.get("ner") or []))))

Full test rows: 25290 | selected: 1000 | by source: {'ai4privacy': 500, 'consentshield_synthetic': 500}
Selected gold span counts: {'CONSENTSHIELD_IDENTITY': 375, 'CONSENTSHIELD_CONTACT': 240, 'CONSENTSHIELD_GOVERNMENT_ID': 49, 'CONSENTSHIELD_FINANCIAL': 49, 'CONSENTSHIELD_CREDENTIAL': 48, 'CONSENTSHIELD_HEALTH': 15}


## Build the detector

Set `CONSENTSHIELD_GLINER_ENABLED=1` and `CONSENTSHIELD_GLINER_MODEL` to a local model directory or model ID to request GLiNER. If it is unavailable or disabled, the factory reports the validator fallback; results will then describe the fallback only.

In [3]:
cfg = GlinerConfig.from_env()
detector, status = build_text_detector(cfg)
validator_detector = ValidatorTextDetector()
GLINER_OK = status.backend == "gliner"
print(json.dumps(status.__dict__, indent=2))
if not GLINER_OK:
    print("GLiNER did not load; GLiNER-specific claims will not be made.")

{
  "backend": "validators",
  "gliner_requested": false,
  "gliner_available": false,
  "reason": "GLiNER disabled (CONSENTSHIELD_GLINER_ENABLED not set)",
  "model_version": "validators-v1"
}
GLiNER did not load; GLiNER-specific claims will not be made.


## Map labels to the dataset ontology and calculate span metrics

The converted dataset uses six broad labels. Detector-specific labels are mapped to those categories only for scoring; the original detector labels are not changed.

In [4]:
PRED_TO_GOLD = {
    "PERSON": "CONSENTSHIELD_IDENTITY", "DATE_OF_BIRTH": "CONSENTSHIELD_IDENTITY",
    "EMAIL_ADDRESS": "CONSENTSHIELD_CONTACT", "PHONE_NUMBER": "CONSENTSHIELD_CONTACT",
    "GOVERNMENT_ID": "CONSENTSHIELD_GOVERNMENT_ID",
    "CREDIT_CARD_NUMBER": "CONSENTSHIELD_FINANCIAL", "BANK_ACCOUNT": "CONSENTSHIELD_FINANCIAL",
    "PASSWORD": "CONSENTSHIELD_CREDENTIAL", "API_KEY": "CONSENTSHIELD_CREDENTIAL",
    "MEDICAL_RECORD": "CONSENTSHIELD_HEALTH", "DIAGNOSIS": "CONSENTSHIELD_HEALTH",
}

def iou(a, b):
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0]))
    union = (a[1]-a[0]) + (b[1]-b[0]) - inter
    return inter / union if union else 0.0

def score_spans(gold, pred, relaxed=False):
    used = set(); tp = 0
    for g in gold:
        for j, p in enumerate(pred):
            if j in used or g["label"] != p["label"]: continue
            exact = g["start"] == p["start"] and g["end"] == p["end"]
            if exact or (relaxed and iou((g["start"],g["end"]),(p["start"],p["end"])) >= 0.5):
                used.add(j); tp += 1; break
    fp = len(pred) - tp; fn = len(gold) - tp
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    return {"gold": len(gold), "predicted": len(pred), "tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": 2*precision*recall/(precision+recall) if precision+recall else 0.0}

def predict_rows(rows, det):
    output = []
    latencies = []
    for r in rows:
        text, gold_raw = reconstruct(r)
        t0 = time.perf_counter(); raw = det.detect(text); latencies.append((time.perf_counter()-t0)*1000)
        pred = []
        for span in raw:
            label = PRED_TO_GOLD.get(span.label)
            if label:
                pred.append({"start": span.start, "end": span.end, "label": label})
        gold = [{"start": g["start"], "end": g["end"], "label": g["label"]} for g in gold_raw]
        output.append({"source": r.get("source", "UNKNOWN"), "gold": gold, "pred": pred})
    return output, latencies

def summarize(items):
    gold = [s for x in items for s in x["gold"]]
    pred = [s for x in items for s in x["pred"]]
    return {"exact": score_spans(gold, pred), "relaxed_iou_0_5": score_spans(gold, pred, relaxed=True), "rows": len(items)}

val_results, val_lat = predict_rows(selected, validator_detector)
report = {"machine": platform.machine(), "dataset": str(DATA.relative_to(ROOT)), "test_rows_total": len(rows),
          "selected_rows": len(selected), "selected_by_source": dict(collections.Counter(r.get("source", "UNKNOWN") for r in selected)),
          "detector_status": status.__dict__, "validators": summarize(val_results), "latency_ms_validators_p50": float(np.percentile(val_lat,50)) if val_lat else None,
          "latency_ms_validators_p95": float(np.percentile(val_lat,95)) if val_lat else None}
if GLINER_OK:
    model_results, model_lat = predict_rows(selected, detector)
    report["gliner"] = summarize(model_results)
    report["latency_ms_gliner_p50"] = float(np.percentile(model_lat,50)) if model_lat else None
    report["latency_ms_gliner_p95"] = float(np.percentile(model_lat,95)) if model_lat else None
else:
    report["gliner"] = {"status": "NOT RUN; detector factory returned validators fallback"}

for source in sorted(set(x["source"] for x in val_results)):
    sub = [x for x in val_results if x["source"] == source]
    report.setdefault("validators_by_source", {})[source] = summarize(sub)
    print(source, json.dumps(report["validators_by_source"][source]["relaxed_iou_0_5"], indent=2))
if GLINER_OK:
    for source in sorted(set(x["source"] for x in model_results)):
        report.setdefault("gliner_by_source", {})[source] = summarize([x for x in model_results if x["source"] == source])
        print("GLiNER", source, json.dumps(report["gliner_by_source"][source]["relaxed_iou_0_5"], indent=2))
print("Validator overall:", json.dumps(report["validators"]["relaxed_iou_0_5"], indent=2))
if GLINER_OK: print("GLiNER overall:", json.dumps(report["gliner"]["relaxed_iou_0_5"], indent=2))

ai4privacy {
  "gold": 505,
  "predicted": 243,
  "tp": 183,
  "fp": 60,
  "fn": 322,
  "precision": 0.7530864197530864,
  "recall": 0.36237623762376237,
  "f1": 0.48930481283422456
}
consentshield_synthetic {
  "gold": 271,
  "predicted": 125,
  "tp": 125,
  "fp": 0,
  "fn": 146,
  "precision": 1.0,
  "recall": 0.4612546125461255,
  "f1": 0.6313131313131314
}
Validator overall: {
  "gold": 776,
  "predicted": 368,
  "tp": 308,
  "fp": 60,
  "fn": 468,
  "precision": 0.8369565217391305,
  "recall": 0.39690721649484534,
  "f1": 0.5384615384615384
}


## Category-level recall and credential coverage

A category with no gold spans cannot be evaluated. In this dataset, credential examples are present in the synthetic split; do not interpret that as external-corpus credential coverage.

In [5]:
def per_category(items, pred_key="pred"):
    labels = sorted(set(s["label"] for x in items for s in x["gold"]))
    out = {}
    for label in labels:
        gold = [s for x in items for s in x["gold"] if s["label"] == label]
        pred = [s for x in items for s in x[pred_key] if s["label"] == label]
        out[label] = score_spans(gold, pred, relaxed=True)
    return out
report["validator_category_metrics"] = per_category(val_results)
if GLINER_OK: report["gliner_category_metrics"] = per_category(model_results)
print("Validator category metrics:", json.dumps(report["validator_category_metrics"], indent=2))
if GLINER_OK: print("GLiNER category metrics:", json.dumps(report["gliner_category_metrics"], indent=2))

# Report credential gold spans by source without exposing text.
credential_coverage = {}
for source in sorted(set(r.get("source", "UNKNOWN") for r in selected)):
    credential_coverage[source] = sum(1 for r in selected for s in (r.get("ner") or [])
                                      if r.get("source", "UNKNOWN") == source and s[2] == "CONSENTSHIELD_CREDENTIAL")
report["credential_gold_spans_by_source"] = credential_coverage
print("Credential gold spans in selected sample:", credential_coverage)

Validator category metrics: {
  "CONSENTSHIELD_CONTACT": {
    "gold": 240,
    "predicted": 283,
    "tp": 235,
    "fp": 48,
    "fn": 5,
    "precision": 0.8303886925795053,
    "recall": 0.9791666666666666,
    "f1": 0.8986615678776291
  },
  "CONSENTSHIELD_CREDENTIAL": {
    "gold": 48,
    "predicted": 53,
    "tp": 48,
    "fp": 5,
    "fn": 0,
    "precision": 0.9056603773584906,
    "recall": 1.0,
    "f1": 0.9504950495049505
  },
  "CONSENTSHIELD_FINANCIAL": {
    "gold": 49,
    "predicted": 7,
    "tp": 2,
    "fp": 5,
    "fn": 47,
    "precision": 0.2857142857142857,
    "recall": 0.04081632653061224,
    "f1": 0.07142857142857142
  },
  "CONSENTSHIELD_GOVERNMENT_ID": {
    "gold": 49,
    "predicted": 25,
    "tp": 23,
    "fp": 2,
    "fn": 26,
    "precision": 0.92,
    "recall": 0.46938775510204084,
    "f1": 0.6216216216216216
  },
  "CONSENTSHIELD_HEALTH": {
    "gold": 15,
    "predicted": 0,
    "tp": 0,
    "fp": 0,
    "fn": 15,
    "precision": 0.0,
    "recall

## Optional adversarial smoke check

This is a detection smoke check only. The JSONL cases contain expected policy actions, not token-level gold spans, so they must not be included in span precision/recall.

In [6]:
if ADV.exists():
    adversarial = read_jsonl(ADV)
    smoke = []
    for case in adversarial:
        stack = [case.get("payload", {})]; string_values = []
        while stack:
            value = stack.pop()
            if isinstance(value, dict): stack.extend(value.values())
            elif isinstance(value, list): stack.extend(value)
            elif isinstance(value, str): string_values.append(value)
        detected = False
        for value in string_values:
            try:
                if detector.detect(value): detected = True; break
            except Exception:
                # Runtime failure must not be counted as a successful clean detection.
                detected = False; break
        smoke.append({"case_id": case.get("case_id"), "expected_action": case.get("expected_action"),
                      "expected_forwarded": case.get("expected_forwarded"), "detector_fired_on_any_string": detected})
    report["adversarial_smoke"] = smoke
    print("Adversarial smoke rows:", len(smoke), "| detector fired:", sum(x["detector_fired_on_any_string"] for x in smoke))
else:
    print("No adversarial file found; skipped smoke check.")

out = RESULTS / "gliner_eval.json"
out.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved", out, "| raw text was not written to report")

Adversarial smoke rows: 12 | detector fired: 5
Saved C:\Users\Chris\Downloads\ibmz2026-main\ibmz2026-main\results\gliner_eval.json | raw text was not written to report
